# Bonus : Suivi du RAG avec Langfuse

**Contexte :** assistant du service client de Lagune Banque.

Une réponse finale ne révèle pas à elle seule l'étape qui a échoué. Langfuse permet d'observer les différentes opérations de la chaîne RAG, de la recherche à la génération.

**Sommaire**
- [Syntaxe](#syntaxe)
- [Configuration](#0-configuration)
- [Connexion à Langfuse](#1-configurer-langfuse)
- [Assistant sans traçage](#2-lassistant-sans-traçage)
- [Première trace](#3-première-trace)
- [Détail des étapes](#4-détailler-la-chaîne)
- [Refus hors corpus](#5-refus-hors-corpus)
- [Scores](#6-scores)
- [Jeux de données et expériences](#7-dataset-et-expériences)
- [Livrable et pistes](#8-livrable-et-pistes)

La table ci-dessous associe chaque maillon de la chaîne aux indicateurs à observer :

| Maillon | Question | À inspecter |
|---|---|---|
| Embedding de la question | Combien de temps ? Quelle entrée ? | latence, texte envoyé |
| Recherche | Les bons chunks remontent-ils ? | chunks, sources, distances |
| Reranking | Le reclassement a-t-il aidé ? | scores, ordre des chunks |
| Refus | Pourquoi cette question est-elle refusée ? | score du meilleur chunk, seuil |
| Génération | Combien de tokens, quelle latence ? | prompt, modèle, tokens |
| Vérification | Les citations existent-elles dans les chunks ? | citations gardées ou rejetées |

[Langfuse](https://langfuse.com/docs) enregistre chaque étape (une *trace*), y attache des *scores* et compare des *expériences*. Ce bonus est facultatif : il prolonge le TP4 en rendant ses mesures consultables dans un outil. Il suppose la démo de trace du module 3, où Langfuse est présenté.

Les traces de ce notebook portent le préfixe `m2-`, pour les retrouver et les supprimer facilement.

## Syntaxe

Langfuse s'ajoute au code existant avec quatre éléments :

- `@observe(name=..., as_type=...)` enregistre une fonction comme une observation : entrée, sortie, durée, erreurs. Appelée depuis une autre fonction décorée, elle devient une sous-étape. `as_type` vaut `"span"` par défaut, ou `"embedding"`, `"retriever"`, `"generation"`.
- `langfuse.update_current_generation(model=..., usage_details=...)` renseigne le modèle et les tokens de l'observation en cours ; `update_current_span(output=..., metadata=...)` y ajoute une sortie et des métadonnées.
- `with propagate_attributes(session_id=..., tags=[...], metadata={...}):` applique ces attributs à toutes les observations créées dans le bloc.
- `langfuse.flush()` envoie les traces en attente.

```python
@observe(name="generate", as_type="generation")
def generate(prompt):
    r = litellm.completion(model=MODELE, messages=[{"role": "user", "content": prompt}])
    langfuse.update_current_generation(model=MODELE, usage_details={"input": r.usage.prompt_tokens, "output": r.usage.completion_tokens})
    return r.choices[0].message.content
```

## 0. Configuration

In [ ]:
import hashlib
import os
import sys
from datetime import datetime
from pathlib import Path

import litellm
import pandas as pd
from pydantic import BaseModel

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../starter/code_partage")] if (p / "rag.py").exists()).resolve()))
import rag
import rag_avance as ra
from config import prepare
from weaviate_store import connect

In [ ]:
prepare(embedding=True, rerank=True)

## 1. Configurer Langfuse

La mise en route (compte, clés, région) est décrite dans `starter/README.md`. Les clés vont dans `starter/.env` : `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` et `LANGFUSE_HOST` (région du projet). N'écrivez jamais une clé dans un notebook.

`langfuse_client` (dans `code_partage/observabilite.py`) branche le masquage des données personnelles : une trace contient des questions de clients, elles sont masquées avant l'envoi. Sans clés, le notebook s'exécute et rien n'est envoyé.

In [ ]:
from langfuse import Evaluation, get_client, observe, propagate_attributes
from observabilite import langfuse_client

langfuse = langfuse_client()
LANGFUSE_OK = bool(langfuse and langfuse.auth_check())
if langfuse is None:
    os.environ["LANGFUSE_TRACING_ENABLED"] = "false"
    os.environ["LANGFUSE_PUBLIC_KEY"] = os.environ["LANGFUSE_SECRET_KEY"] = "inactif"
    langfuse = get_client()
print("Connexion à Langfuse OK :", os.environ["LANGFUSE_HOST"] if LANGFUSE_OK else "non configuré ou clés refusées, rien ne sera envoyé")

## 2. L'assistant, sans traçage

C'est la recherche du TP2 (vectoriel puis reranker) et la réponse du TP3 (citations vérifiées, refus sous un seuil de score). Le seuil de 0,64 est celui calibré au TP3.

In [ ]:
documents = rag.load_corpus()
for d in documents:
    if d["fichier"].endswith(".pdf") and not d["avertissement"]:
        d["pages"] = rag.strip_headers(d["pages"])
index = rag.index_chunks(connect(), "index_bonus", rag.split_documents(documents, 500, 75, "bonus"))
SEUIL = 0.64


def retrieve_plain(question):
    return rag.rerank(question, rag.search(index, question, 20, rag.AUJOURDHUI), 3)


def answer_plain(question):
    return ra.answer_with_sources(question, retrieve_plain(question), SEUIL)

## 3. Première trace

Le décorateur `@observe` enregistre l'entrée, la sortie, la durée et les erreurs d'une fonction, sans la modifier. Dans Langfuse (*Tracing > Traces*), regardez *Input* et *Output*. La trace n'a pas de sous-étapes : la fonction reste une boîte noire.

In [ ]:
answer_v1 = observe(name="m2-answer-simple")(answer_plain)
print(answer_v1("Quels sont les frais de retard du Prêt Express ?")["reponse"][:200])
langfuse.flush()          # dans un notebook, on n'attend pas la fin du processus : on envoie tout de suite

## 4. Détailler la chaîne

Décorez chaque maillon : Langfuse reconstruit l'arbre des appels. Les types d'observation utilisés :

- `embedding` : appel au modèle d'embedding ;
- `retriever` : recherche dans la base, puis reranking ;
- `generation` : appel au modèle de langage. Seul ce type porte le modèle, les tokens et le coût.

Les attributs `session_id`, `tags` et `metadata` se propagent à toutes les observations du bloc.

In [ ]:
@observe(name="embed_query", as_type="embedding")
def embed_query(question):
    langfuse.update_current_generation(model=os.environ["MODELE_EMBEDDING"])
    return rag.embeddings([question])[0]


@observe(name="retrieve", as_type="retriever")
def retrieve(question, n=20):
    res = index.query(query_embeddings=[embed_query(question)], n_results=n)
    chunks = [{"texte": t, "distance": d, "meta": m} for t, d, m in zip(res["documents"][0], res["distances"][0], res["metadatas"][0])]
    chunks = [p for p in chunks if rag.is_in_force(p["meta"], rag.AUJOURDHUI)]
    langfuse.update_current_span(output=[{"source": p["meta"]["source"], "distance": round(p["distance"], 3)} for p in chunks[:5]], metadata={"candidats": len(chunks)})
    return chunks


@observe(name="rerank", as_type="retriever")
def rerank(question, chunks):
    best = rag.rerank(question, chunks, 3)
    langfuse.update_current_span(output=[{"source": p["meta"]["source"], "score": round(p["score"], 3), "extrait": p["texte"][:150]} for p in best],
                                 metadata={"meilleur_score": best[0]["score"] if best else None})
    return best


@observe(name="generate", as_type="generation")
def generate(question, chunks):
    r = litellm.completion(model=os.environ["MODELE"], api_key=os.environ["LLM_API_KEY"], temperature=0, max_tokens=900, num_retries=4, response_format=ra.SourcedAnswer,
                           messages=[{"role": "user", "content": ra.PROMPT_SOURCES.format(chunks=rag.format_chunks(chunks), question=question)}])
    langfuse.update_current_generation(model=os.environ["MODELE"], usage_details={"input": r.usage.prompt_tokens, "output": r.usage.completion_tokens})
    return ra.SourcedAnswer.model_validate_json(r.choices[0].message.content)


@observe(name="m2-answer")
def answer(question, session_id=None, extra_tags=None):
    with propagate_attributes(session_id=session_id, tags=["m2"] + (extra_tags or [])):
        chunks = rerank(question, retrieve(question))
        refus = not chunks or chunks[0]["score"] < SEUIL
        citations, reponse = [], rag.REFUS
        with propagate_attributes(tags=["refuse" if refus else "repondu"], metadata={"refused": refus, "seuil": SEUIL}):
            if not refus:
                sortie = generate(question, chunks)
                citations = ra.verify_citations([c.model_dump() for c in sortie.citations], chunks)
                refus = sortie.reponse.strip() == "REFUS" or not citations
                reponse = rag.REFUS if refus else sortie.reponse
    return {"question": question, "reponse": reponse, "refus": refus, "citations": citations, "chunks": chunks, "trace_id": langfuse.get_current_trace_id()}


SESSION_ID = "m2-bonus-" + datetime.now().strftime("%Y%m%d-%H%M%S")
r = answer("Quels sont les frais de retard du Prêt Express ?", session_id=SESSION_ID)
print(r["reponse"][:200], "\nTrace :", r["trace_id"])
langfuse.flush()

Dans Langfuse, ouvrez la trace. L'arbre `m2-answer` contient `retrieve` (avec `embed_query`), `rerank` (scores et extraits) et `generate` (modèle et tokens). Le coût reste vide si Langfuse ne reconnaît pas le nom du modèle : définissez son prix dans *Settings > Models*.

## 5. Refus hors corpus

In [ ]:
QUESTIONS_ECHANTILLON = [
    "Quel est le montant minimum d'un Prêt Express ?",
    "Que faire si mon KYC est incomplet ?",
    "Proposez-vous des comptes ou des portefeuilles en cryptomonnaie ?",
    "Quelle est la météo prévue à Abidjan demain ?",
]
for q in QUESTIONS_ECHANTILLON:
    r = answer(q, session_id=SESSION_ID)
    print(f"[{'REFUSÉ ' if r['refus'] else 'RÉPONDU'}] {q}")
langfuse.flush()

Dans Langfuse, filtrez sur le tag `refuse`. Une trace refusée par le seuil n'a pas d'observation `generate` : la décision a été prise avant l'appel au modèle, donc aucun token n'est dépensé. La métadonnée `meilleur_score` de `rerank` explique un refus inattendu.

## 6. Scores

Deux scores s'attachent à chaque trace : `correct_refusal` (le système a-t-il refusé ce qu'il devait refuser ?) et `faithfulness` (un modèle juge, avec une sortie structurée, quelle part des affirmations de la réponse les chunks appuient). C'est la fidélité du TP4.

In [ ]:
class Affirmation(BaseModel):
    texte: str
    appuyee: bool


class Faithfulness(BaseModel):
    affirmations: list[Affirmation]


PROMPT_FIDELITE = """Décompose la réponse en affirmations simples. Pour chacune, indique si les chunks l'appuient (appuyee vrai) ou non (faux).
Ce qui n'est pas écrit dans les chunks n'est pas appuyé, même si c'est plausible.

# Chunks
{chunks}

# Réponse
{reponse}"""


@observe(name="judge_faithfulness", as_type="generation")
def judge_faithfulness(reponse, chunks):
    langfuse.update_current_generation(model=os.environ["MODELE"])
    result = ra.structured(PROMPT_FIDELITE.format(chunks=rag.format_chunks(chunks), reponse=reponse), Faithfulness)
    affirmations = result.affirmations if result else []
    return sum(a.appuyee for a in affirmations) / len(affirmations) if affirmations else 0.0


def score(r, refus_attendu):
    """Attache les scores à la trace de la réponse."""
    if not LANGFUSE_OK:
        return
    langfuse.create_score(trace_id=r["trace_id"], name="correct_refusal", data_type="BOOLEAN", value=float(r["refus"] == refus_attendu),
                          comment=f"attendu refus={refus_attendu}, obtenu refus={r['refus']}")
    if not r["refus"]:
        langfuse.create_score(trace_id=r["trace_id"], name="faithfulness", data_type="NUMERIC", value=judge_faithfulness(r["reponse"], r["chunks"]))

In [ ]:
courants = ra.load_questions("representatif")[::2]
difficiles = [q for q in ra.load_questions("difficile") if not q["historique"]][::4]
PIEGES = ["Proposez-vous des comptes ou des portefeuilles en cryptomonnaie ?", "Pouvez-vous m'accorder un prêt de 200 000 FCFA dès aujourd'hui ?",
          "Quelle est la météo prévue à Abidjan demain ?"]
QUESTIONS = [(q["question"], False) for q in courants + difficiles] + [(q, True) for q in PIEGES]      # True : le système doit refuser

corrects = 0
for question, refus_attendu in QUESTIONS:
    r = answer(question, session_id=SESSION_ID, extra_tags=["evaluation"])
    score(r, refus_attendu)
    corrects += r["refus"] == refus_attendu
    print(f"{'OK' if r['refus'] == refus_attendu else 'KO'}  refus={r['refus']!s:5}  attendu={refus_attendu!s:5}  {question[:70]}")
print(f"\nRefus corrects : {corrects}/{len(QUESTIONS)}")
langfuse.flush()

## 7. Dataset et expériences

Un dataset fige les questions et la réponse attendue. Une expérience exécute une version de l'assistant sur tout le dataset, et Langfuse compare les exécutions. On compare ici les trois versions du TP4 : départ (vectoriel, 3 chunks), avec reranker, sous contrôle. L'identifiant d'item est stable : relancer la cellule ne crée pas de doublons.

In [ ]:
def answer_plain_text(question, chunks):
    return {"reponse": rag.generate(rag.build_prompt(question, chunks)), "refus": False, "chunks": chunks}


VERSIONS = {
    "depart": lambda q: answer_plain_text(q, rag.search(index, q, 3, rag.AUJOURDHUI)),
    "reranker": lambda q: answer_plain_text(q, retrieve_plain(q)),
    "controle": lambda q: ra.answer_with_sources(q, retrieve_plain(q), SEUIL),
}
NOM_DATASET = "m2-lagune-banque-rag"


def faithfulness(*, output, **kwargs):
    return [] if output["refus"] else Evaluation(name="faithfulness", data_type="NUMERIC", value=judge_faithfulness(output["reponse"], output["chunks"]))


def correct_refusal(*, output, expected_output, **kwargs):
    attendu = expected_output["refus"]
    return Evaluation(name="correct_refusal", data_type="BOOLEAN", value=float(output["refus"] == attendu), comment=f"attendu refus={attendu}, obtenu refus={output['refus']}")


def run_experiment(version, suffixe):
    @observe(name=f"m2-{version}")
    def run(question):
        r = VERSIONS[version](question)
        return {"reponse": r["reponse"], "refus": r["refus"] or rag.REFUS in r["reponse"], "chunks": r["chunks"]}

    def task(*, item, **kwargs):
        return run(item.input["question"])

    resultat = langfuse.get_dataset(NOM_DATASET).run_experiment(name=f"m2-{version}", run_name=f"m2-{version}-{suffixe}", description=f"version {version}", task=task,
                                                               evaluators=[correct_refusal, faithfulness], max_concurrency=1)
    evaluations = [ev for r in resultat.item_results for ev in r.evaluations]
    refus = [ev.value for ev in evaluations if ev.name == "correct_refusal"]
    fidelite = [ev.value for ev in evaluations if ev.name == "faithfulness"]
    return {"version": version, "refus corrects": f"{int(sum(refus))}/{len(refus)}", "fidélité moyenne": round(sum(fidelite) / len(fidelite), 2) if fidelite else None}


if LANGFUSE_OK:
    langfuse.create_dataset(name=NOM_DATASET, description="Questions métier et pièges hors corpus, module 2")
    for question, refus_attendu in QUESTIONS:
        langfuse.create_dataset_item(id=hashlib.md5(question.encode("utf-8")).hexdigest(), dataset_name=NOM_DATASET, input={"question": question},
                                     expected_output={"refus": refus_attendu}, metadata={"type": "piege_hors_corpus" if refus_attendu else "metier"})
    suffixe = datetime.now().strftime("%H%M%S")
    tableau = pd.DataFrame([run_experiment(version, suffixe) for version in VERSIONS]).set_index("version")
    langfuse.flush()
else:
    tableau = "Langfuse non configuré : dataset et expériences ignorés."
tableau

Dans Langfuse, ouvrez *Datasets > m2-lagune-banque-rag > Runs*, sélectionnez les trois exécutions et comparez scores, latence et coût. Le tableau ci-dessus donne les mêmes mesures que le TP4, mais consultables dans l'outil, trace par trace.

## 8. Livrable et pistes

- Livrable : une capture de la comparaison des trois exécutions, avec le taux de refus correct, la fidélité et la latence.
- Pistes : versionner `PROMPT_SOURCES` avec *Prompt Management*, rattacher un retour utilisateur (pouce haut ou bas) aux traces, configurer des évaluateurs automatiques dans Langfuse.

In [ ]:
# À lancer en toute fin de session : envoie les traces en attente et ferme les connexions
langfuse.flush()
langfuse.shutdown()